To force a PySpark notebook to create a new Spark session with custom configurations, rather than using an existing or default one, follow these steps:

- **Stop any existing Spark Session:** If a Spark session is already active in your notebook environment (e.g., from a previous run or automatic startup), you must stop it before creating a new one. This ensures that the new session is truly independent and applies your custom configurations from scratch.

In [1]:
if 'spark' in locals() and spark is not None:
    spark.stop()

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, avg, desc, col

spark = SparkSession.builder \
      .appName("medals_matches_homework") \
      .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
      .getOrCreate()

spark

In [3]:
SHARED_PATH = "/home/iceberg/notebooks/notebooks/data"

match_details = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/match_details.csv")
matches = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/matches.csv")
medals_matches_players = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/medals_matches_players.csv")
medals = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/medals.csv")
maps = spark.read.option("header","true").option("inferSchema", "true").csv(f"{SHARED_PATH}/maps.csv")

print(f"{match_details.count()}, {matches.count()}, {medals_matches_players.count()}, {medals.count()}, {maps.count()}")

151761, 24025, 755229, 183, 40


In [4]:
medals_joined = medals_matches_players.join(broadcast(medals), "medal_id")

medals_joined.head()

Row(medal_id=3261908037, match_id='009fdac5-e15c-47c6-a202-e18ff8800ce7', player_gamertag='EcZachly', count=7, sprite_uri='https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png', sprite_left=375, sprite_top=525, sprite_sheet_width=74, sprite_sheet_height=74, sprite_width=1125, sprite_height=899, classification='WeaponProficiency', description='Kill an opponent by shooting them in the head.', name='Headshot', difficulty=60)

In [5]:
matches_joined = matches.join(broadcast(maps), "mapid")

matches_joined.head()

Row(mapid='c7edbf0f-f206-11e4-aa52-24be05e24f7e', match_id='11de1a94-8d07-4162-9f5f-d3cc753c811c', is_team_game=True, playlist_id='f72e0ef0-7c4a-4307-af78-8e38dac3fdba', game_variant_id='1e473914-46e4-408d-af26-178fb115de76', is_match_over=True, completion_date=datetime.datetime(2016, 2, 22, 0, 0), match_duration=None, game_mode=None, map_variant_id=None, name='Breakout Arena', description='The broadcast of Breakout matches has proven immensely popular with the UNSC Infinity crew.')

In [6]:
spark.sql("DROP TABLE IF EXISTS bootcamp.bucketed_match_details")
ddl_match_details = """
CREATE TABLE IF NOT EXISTS bootcamp.bucketed_match_details (
     match_id STRING,
     player_gamertag STRING,
     player_total_kills INTEGER,
     player_total_deaths INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(ddl_match_details)

spark.sql("DROP TABLE IF EXISTS bootcamp.bucketed_matches")
ddl_matches = """
CREATE TABLE IF NOT EXISTS bootcamp.bucketed_matches (
     match_id STRING,
     mapid STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     completion_date TIMESTAMP
 )
USING iceberg
PARTITIONED BY (completion_date, bucket(16, match_id));
"""
spark.sql(ddl_matches)

spark.sql("DROP TABLE IF EXISTS bootcamp.bucketed_medal_matches_players")
ddl_match_details = """
CREATE TABLE IF NOT EXISTS bootcamp.bucketed_medal_matches_players (
     match_id STRING,
     player_gamertag STRING,
     medal_id STRING,
     count INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(ddl_match_details)

DataFrame[]

In [7]:
# Saving DataFrames as bucketed tables
match_details.select("match_id", "player_gamertag", "player_total_kills", "player_total_deaths") \
    .write.mode("overwrite") \
    .bucketBy(16, "match_id") \
    .saveAsTable("bootcamp.bucketed_match_details")

matches.select("match_id", "mapid", "is_team_game", "playlist_id", "completion_date") \
    .write.mode("overwrite") \
    .partitionBy("completion_date") \
    .bucketBy(16, "match_id") \
    .saveAsTable("bootcamp.bucketed_matches")

medals_matches_players.write.mode("overwrite") \
    .bucketBy(16, "match_id") \
    .saveAsTable("bootcamp.bucketed_medal_matches_players")

# Reading the bucketed tables
bucketed_match_details = spark.table("bootcamp.bucketed_match_details")
bucketed_matches = spark.table("bootcamp.bucketed_matches")
bucketed_medal_matches_players = spark.table("bootcamp.bucketed_medal_matches_players")

In [20]:
# The join will automatically use the bucketing since all tables are bucketed
joined_df = bucketed_match_details.alias("md") \
    .join(bucketed_matches.alias("m"), "match_id") \
    .join(bucketed_medal_matches_players.alias("mmp"), "match_id") \
    .select(
        "match_id",
        "md.player_gamertag",  # Explicitly getting value from match_details (duplicate column)
        "player_total_kills",
        "player_total_deaths",
        "mapid",
        "is_team_game",
        "playlist_id",
        "medal_id",
        "count",
        "completion_date"
    )

joined_df.head()

Row(match_id='00169217-cca6-4b47-8df0-559ee424143f', player_gamertag='King Terror V', player_total_kills=14, player_total_deaths=7, mapid='cc040aa1-f206-11e4-a3e0-24be05e24f7e', is_team_game=True, playlist_id='2323b76a-db98-4e03-aa37-e171cfbdd1a4', medal_id=3261908037, count=11, completion_date=datetime.datetime(2016, 3, 13, 0, 0))

In [21]:
# Question 1 - Which player averages the most kills per game?
player_avg_kills = joined_df.groupBy("player_gamertag") \
    .agg(avg("player_total_kills").alias("avg_kills_per_game")) \
    .orderBy(desc("avg_kills_per_game"))

player_avg_kills.head(3)

[Row(player_gamertag='gimpinator14', avg_kills_per_game=109.0),
 Row(player_gamertag='I Johann117 I', avg_kills_per_game=96.0),
 Row(player_gamertag='BudgetLegendary', avg_kills_per_game=83.0)]

In [10]:
# Question 2 - Which playlist gets played the most?
executed_playlists = joined_df.groupBy("playlist_id") \
    .count() \
    .withColumnRenamed("count", "execution_count") \
    .orderBy(desc("execution_count"))

executed_playlists.head(3)

[Row(playlist_id='f72e0ef0-7c4a-4307-af78-8e38dac3fdba', execution_count=1565529),
 Row(playlist_id='780cc101-005c-4fca-8ce7-6f36d7156ffe', execution_count=1116002),
 Row(playlist_id='0bcf2be1-3168-4e42-9fb5-3551d7dbce77', execution_count=1015496)]

In [11]:
# Question 3 - Which map gets played the most?
played_maps = joined_df.groupBy("mapid") \
    .count() \
    .withColumnRenamed("count", "played_count") \
    .orderBy(desc("played_count"))

played_maps.head(3)

[Row(mapid='c74c9d0f-f206-11e4-8330-24be05e24f7e', played_count=1445545),
 Row(mapid='c7edbf0f-f206-11e4-aa52-24be05e24f7e', played_count=1435048),
 Row(mapid='c7805740-f206-11e4-982c-24be05e24f7e', played_count=953278)]

In [16]:
# Question 4 - Which map do players get the most Killing Spree medals on?
most_killing_spree_maps = joined_df \
    .join(medals, "medal_id") \
    .where(col("name") == "Killing Spree") \
    .groupBy("mapid") \
    .count() \
    .withColumnRenamed("count", "medal_count") \
    .orderBy(desc("medal_count"))

most_killing_spree_maps.head(3)

[Row(mapid='c74c9d0f-f206-11e4-8330-24be05e24f7e', medal_count=56908),
 Row(mapid='c7edbf0f-f206-11e4-aa52-24be05e24f7e', medal_count=50570),
 Row(mapid='c7805740-f206-11e4-982c-24be05e24f7e', medal_count=35444)]

#### With the aggregated dataset, testing different .sortWithinPartitions() to see which has the smallest data size (hint: playlists and maps are both very low cardinality)

In [17]:
spark.sql("DROP TABLE IF EXISTS bootcamp.sorted_matches")
ddl_sorted_matches = """
CREATE TABLE IF NOT EXISTS bootcamp.sorted_matches (
     match_id STRING,
     player_gamertag STRING,
     player_total_kills INTEGER,
     player_total_deaths INTEGER,
     mapid STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     medal_id STRING,
     count INTEGER,
     completion_date TIMESTAMP
 )
USING iceberg
PARTITIONED BY (year(completion_date));
"""
spark.sql(ddl_sorted_matches)

DataFrame[]

In [22]:
joined_df \
    .sortWithinPartitions(col("completion_date"), col("mapid")) \
    .write.mode("overwrite") \
    .saveAsTable("bootcamp.sorted_matches")

In [23]:
%%sql

SELECT SUM(file_size_in_bytes) as file_size_in_bytes, COUNT(1) as num_files 
FROM demo.bootcamp.sorted_matches.files

25/09/11 21:17:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


file_size_in_bytes,num_files
6347935,15


In [24]:
joined_df \
    .sortWithinPartitions(col("completion_date"), col("playlist_id")) \
    .write.mode("overwrite") \
    .saveAsTable("bootcamp.sorted_matches")

In [25]:
%%sql

SELECT SUM(file_size_in_bytes) as file_size_in_bytes, COUNT(1) as num_files 
FROM demo.bootcamp.sorted_matches.files

file_size_in_bytes,num_files
6330330,15


In [26]:
joined_df \
    .sortWithinPartitions(col("completion_date"), col("playlist_id"), col("mapid")) \
    .write.mode("overwrite") \
    .saveAsTable("bootcamp.sorted_matches")

In [27]:
%%sql

SELECT SUM(file_size_in_bytes) as file_size_in_bytes, COUNT(1) as num_files 
FROM demo.bootcamp.sorted_matches.files

file_size_in_bytes,num_files
6339039,15


In [28]:
joined_df \
    .sortWithinPartitions(col("completion_date"), col("mapid"), col("playlist_id")) \
    .write.mode("overwrite") \
    .saveAsTable("bootcamp.sorted_matches")

In [29]:
%%sql

SELECT SUM(file_size_in_bytes) as file_size_in_bytes, COUNT(1) as num_files 
FROM demo.bootcamp.sorted_matches.files

file_size_in_bytes,num_files
6348994,15
